In [1]:
from sentence_tokenization import *
from tf import Transformer
import numpy as np

In [2]:
max_sequence_length = 200
NEG_INFTY = -1e9

english_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/train_200k.en'
chinese_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/train_200k.zh'
# english_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/UNv1.0.en-zh.en'
# chinese_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/UNv1.0.en-zh.zh'

START_TOKEN = '<START>'
PADDING_TOKEN = '<PADDING>'
END_TOKEN = '<END>'

english_vocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.',
                      '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
                      ':', '<', '=', '>', '?', '@',
                      'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L',
                      'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X',
                      'Y', 'Z',
                      '[', '\\', ']', '^', '_', '`',
                      'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
                      'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 
                      'y', 'z','?',
                      '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

chinese_vocabulary = [START_TOKEN, ' ', '！', '“', "”", '#', '$', '%', '&', "’", "‘", '（', '）', '*', '+', 
                    '，', '-', '。', '/',  
                    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '：', '<', '=', '>', '？', 
                    PADDING_TOKEN, END_TOKEN]

In [3]:
english_vocabulary, english_to_index, index_to_english = build_eng_vocab(english_file)
chinese_vocabulary, chinese_to_index, index_to_chinese = build_zh_vocab(chinese_file)

In [4]:
# 1. 初始化
english_embedding = SentenceEmbedding(
    max_sequence_length=200,
    d_model=512,
    language_to_index=english_to_index,
    language='en',
    START_TOKEN=START_TOKEN,
    END_TOKEN=END_TOKEN,
    PADDING_TOKEN=PADDING_TOKEN
)

chinese_embedding = SentenceEmbedding(
    max_sequence_length=200,
    d_model=512,
    language_to_index=chinese_to_index,
    language='zh',
    START_TOKEN=START_TOKEN,
    END_TOKEN=END_TOKEN,
    PADDING_TOKEN=PADDING_TOKEN
)

english_embedding = english_embedding.to(get_device())
chinese_embedding = chinese_embedding.to(get_device())

In [5]:
d_model = 512
batch_size = 32
ffn_hidden = 2048
num_heads = 8
drop_prob = 0.1
num_layers = 1
max_sequence_length = 200
zh_vocab_size = len(chinese_vocabulary)

transformer = Transformer(d_model, 
                          ffn_hidden,
                          num_heads, 
                          drop_prob, 
                          num_layers, 
                          max_sequence_length,
                          zh_vocab_size,
                          english_to_index,
                          chinese_to_index,
                          START_TOKEN, 
                          END_TOKEN, 
                          PADDING_TOKEN)

In [6]:
transformer

Transformer(
  (encoder): Encoder(
    (sentence_embedding): SentenceEmbedding(
      (embedding): Embedding(46373, 512)
      (position_encoder): PositionalEncoding()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): SequentialEncoder(
      (0): EncoderLayer(
        (attention): MutliHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): Normalnize()
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): PostitionwiseFeedForward(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (relu): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): Normalnize()
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Decoder(
    (sentence_embedding): SentenceEmb

In [7]:
# 2. 准备输入数据
with open(english_file, 'r') as file:
    english_sentences = file.readlines()
with open(chinese_file, 'r') as file:
    chinese_sentences = file.readlines()

# Limit Number of sentences ## 15886041
#                                200000
TOTAL_SENTENCES = 190000
english_sentences = english_sentences[:TOTAL_SENTENCES]
chinese_sentences = chinese_sentences[:TOTAL_SENTENCES]
english_sentences = [sentence.rstrip('\n') for sentence in english_sentences]
chinese_sentences = [sentence.rstrip('\n') for sentence in chinese_sentences]

# valid_sentence_indicies = []
# for index in range(len(chinese_sentences)):
#     chinese_sentence, english_sentence = chinese_sentences[index], english_sentences[index]
#     if is_valid_zh_length(chinese_sentence, max_sequence_length=max_sequence_length) \
#         and is_valid_eng_length(english_sentence, max_sequence_length=max_sequence_length) \
#         and is_valid_tokens(chinese_sentence, chinese_vocabulary):
#             valid_sentence_indicies.append(index)
    
# chinese_sentences = [chinese_sentences[i] for i in valid_sentence_indicies]
# english_sentences = [english_sentences[i] for i in valid_sentence_indicies]


In [8]:
english_sentences[:3]

['RESOLUTION 918 (1994)',
 'Adopted by the Security Council at its 3377th meeting, on 17 May 1994',
 'The Security Council,']

In [9]:
chinese_sentences[:3]

['第918(1994)号决议', '1994年5月17日安全理事会第3377次会议通过', '安全理事会，']

In [10]:
# dataset = TextDataset()
dataset = TextDataset(english_sentences, chinese_sentences)

train_loader = DataLoader(dataset, batch_size)

In [11]:
from torch import nn

criterian = nn.CrossEntropyLoss(ignore_index=chinese_to_index[PADDING_TOKEN],
                                reduction='none')

# When computing the loss, we are ignoring cases when the label is the padding token
for params in transformer.parameters():
    if params.dim() > 1:
        nn.init.xavier_uniform_(params)

optim = torch.optim.Adam(transformer.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [12]:
transformer.train()
transformer.to(device)
total_loss = 0
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    iterator = iter(train_loader)
    for batch_num, batch in enumerate(iterator):
        transformer.train()
        eng_batch, zh_batch = batch
        encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask = create_masks(eng_batch, zh_batch)
        optim.zero_grad()
        zh_predictions = transformer(eng_batch,
                                     zh_batch,
                                     encoder_self_attention_mask.to(device), 
                                     decoder_self_attention_mask.to(device), 
                                     decoder_cross_attention_mask.to(device),
                                     enc_start_token=False,
                                     enc_end_token=False,
                                     dec_start_token=True,
                                     dec_end_token=False)
        labels = transformer.decoder.sentence_embedding.batch_tokenize(zh_batch, start_token=False, end_token=True)
        loss = criterian(
            zh_predictions.view(-1, zh_vocab_size).to(device),
            labels.view(-1).to(device)
        ).to(device)
        valid_indicies = torch.where(labels.view(-1) == chinese_to_index[PADDING_TOKEN], False, True)
        loss = loss.sum() / valid_indicies.sum()
        loss.backward()
        optim.step()
        #train_losses.append(loss.item())
        if batch_num % 1000 == 0:
            print(f"Iteration {batch_num} : {loss.item()}")
            print(f"English: {eng_batch[0]}")
            print(f"chinese Translation: {zh_batch[0]}")
            zh_sentence_predicted = torch.argmax(zh_predictions[0], axis=1)
            predicted_sentence = ""
            for idx in zh_sentence_predicted:
                if idx == chinese_to_index[END_TOKEN]:
                    break
                predicted_sentence += index_to_chinese[idx.item()]
            print(f"chinese Prediction: {predicted_sentence}")

            transformer.eval()
            zh_sentence = ("",)
            eng_sentence = ("The use of remote sensing satellite data.",)
            for word_counter in range(max_sequence_length):
                encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, zh_sentence)
                predictions = transformer(eng_sentence,
                                          zh_sentence,
                                          encoder_self_attention_mask.to(device), 
                                          decoder_self_attention_mask.to(device), 
                                          decoder_cross_attention_mask.to(device),
                                          enc_start_token=False,
                                          enc_end_token=False,
                                          dec_start_token=True,
                                          dec_end_token=False)
                next_token_prob_distribution = predictions[0][word_counter] # not actual probs
                next_token_index = torch.argmax(next_token_prob_distribution).item()
                next_token = index_to_chinese[next_token_index]
                zh_sentence = (zh_sentence[0] + next_token, )
                if next_token == END_TOKEN:
                  break
            
            print(f"Evaluation translation (The use of remote sensing satellite data.) : {zh_sentence}")
            
            print("-------------------------------------------")
    
    torch.save(
        transformer.state_dict(),
        f"checkpoint_epoch_{epoch}.pth"
    )


Epoch 0
Iteration 0 : 8.549960136413574
English: RESOLUTION 918 (1994)
chinese Translation: 第918(1994)号决议
chinese Prediction: 伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙缘伙伙厅伙伙伙伙甚伙伙伙伙伙伙伙伙伙鞑请请伙伙伙伙稿伙稿伙伙伙伙伙伙伙伙伙伙稿庭伙逮伙伙伙伙伙伙伙伙伙伙娴伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙鞑伙伙伙伙珀伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙甚伙伙伙伙伙伙甚伙伙缘稿您稿伙缘伙伙伙伙伙甚伙甚伙伙伙伙稿革逮伙约伙伙伙稿伙伙伙珀缘伙约伙伙伙稿甚稿捧庭革伙稿稿
Evaluation translation (The use of remote sensing satellite data.) : ('伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙逮逮逮逮伙伙伙伙伙伙伙伙伙逮逮伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙逮逮逮逮伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙伙',)
-------------------------------------------
Iteration 1000 : 5.745436191558838
English: The reduction of the export refunds and of the subsidized export quotas will increase the chances of the developing countries becoming more actively involved in the world market in future.
chinese Translation: 减少出口退税和出口配额补贴将会增加发展中国家今后积极参与世界市场的机会。
chinese Prediction: ((在的的的的在的的的的的的的的的的的的的的的的的的的的的的的的欧的欧欧
Evalua

In [ ]:
# 保存优化器

# 推荐保存：

# torch.save({
#     "epoch": epoch,
#     "model_state_dict": transformer.state_dict(),
#     "optimizer_state_dict": optim.state_dict(),
# }, "checkpoint.pth")

# 恢复训练：

# checkpoint = torch.load("checkpoint.pth")

# transformer.load_state_dict(
#     checkpoint["model_state_dict"]
# )

# optim.load_state_dict(
#     checkpoint["optimizer_state_dict"]
# )

# start_epoch = checkpoint["epoch"] + 1

# 这样可以从断点继续训练，而不是重新开始。


# transformer = Transformer(
#     d_model=d_model,
#     ffn_hidden=ffn_hidden,
#     num_heads=num_heads,
#     drop_prob=drop_prob,
#     num_layers=num_layers,
#     max_sequence_length=max_sequence_length,
#     zh_vocab_size=zh_vocab_size,
#     english_to_index=english_to_index,
#     chinese_to_index=chinese_to_index,
#     START_TOKEN=START_TOKEN,
#     END_TOKEN=END_TOKEN,
#     PADDING_TOKEN=PADDING_TOKEN
# )
# checkpoint = torch.load(
#     "checkpoint_epoch_3.pth",
#     map_location=device
# )

# transformer.load_state_dict(checkpoint)

# transformer.to(device)
# transformer.eval()

In [13]:
transformer.eval()
def translate(eng_sentence):
  eng_sentence = (eng_sentence,)
  zh_sentence = ("",)
  for word_counter in range(max_sequence_length):
    encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, zh_sentence)
    predictions = transformer(eng_sentence,
                              zh_sentence,
                              encoder_self_attention_mask.to(device), 
                              decoder_self_attention_mask.to(device), 
                              decoder_cross_attention_mask.to(device),
                              enc_start_token=False,
                              enc_end_token=False,
                              dec_start_token=True,
                              dec_end_token=False)
    next_token_prob_distribution = predictions[0][word_counter]
    next_token_index = torch.argmax(next_token_prob_distribution).item()
    next_token = index_to_chinese[next_token_index]
    zh_sentence = (zh_sentence[0] + next_token, )
    if next_token == END_TOKEN:
      break
  return zh_sentence[0]

In [14]:
translation = translate("So far, relief has still been the main form of disaster management.")
print(translation)
#迄今为止，救济仍然是灾害管理的主要形式

这是一种形式的重大量有限制性质的影响欧<END>


In [15]:
translation = translate("The data distribution system of these satellites will be greatly improved compared to now.")
print(translation)
#这些卫星的数据分配系统将比现在有大大的改善。

这些学习的数据库是重这些学生的资料是一种欧<END>


In [16]:
translation = translate("They realized that the biggest problems are economic and political issues.")
print(translation)
#他们认识到，最大的问题是经济和政治问题。

这些问题是经济和政治问题的政治意见欧<END>


In [17]:
translation = translate("These manuals are distributed nationwide.")
print(translation)
#这些手册在全国发行。

这些数据库包括一个国家欧<END>
